Generated at: 2026-06-01 17:37:04 MSK

# ML-005 design notebook

- service: NBO scoring / top-10 next-best-offer for B2B software license vendor
- public domain: `client_id`, `PROD_01..PROD_39`, `SEG_0..SEG_6`
- declared maturity: **Level 2**
- core idea: batch precompute + cached serving API
- notebook is readable without running code

**Вывод:** this notebook is the homework-facing design summary, not an experiment log.


## 1. Architecture comparison

| pattern | latency | freshness | cost | complexity | fit for NBO |
|---|---|---|---|---|---|
| batch precompute top-N quarterly | low at request time, cache lookup | quarter-level / delayed Q+1 labels | low | medium | best fit: catalog small, sales workflow can use batch freshness |
| reactive / event-driven retrain | medium, depends on trigger | fresh after batch event | medium | medium | good for CT loop: sensor -> validate -> train -> gate |
| online request-time compute | high, feature compute in request path | freshest | medium | high | weak fit: latency risk, harder parity, MDD rejects it |
| streaming Kafka/Spark | low to medium | near real-time | high | high | overengineering for quarterly NBO and 1 VM demo |
| HITL approval only | human latency | depends on operator | medium | low to medium | useful as governance layer, but not enough for Level 2 automation |

**Вывод:** выбран hybrid: batch precompute + reactive retraining trigger + on-demand single-client API from cache.


## 2. Chosen architecture

Flow:

```text
events -> PIT features -> train champion/challenger -> evaluate -> MLflow alias -> FastAPI /score
```

Chosen path:

- batch precompute top-N / feature store / registry
- on-demand single-client scoring through API
- no live remote sync in request path
- Airflow owns retraining and metric gate
- CI owns compile / smoke / Terraform plan, not training

Why not streaming / k8s:

- quarterly horizon Q+1 -> streaming freshness is not needed for this homework target
- one local docker stack / future 1 VM is enough
- risk register says overengineering is a real risk
- MDD latency result supports cache-based serving

Evidence:

- API: `../src/serve_api.py`
- registry: `../src/registry.py`
- feature contract: `../artifacts/feature_list.json`
- ADR: `../adr/0001-latency-mdd-decision.md`


## 3. DAG design

![DAG graph target](../screenshots/b07_dag_graph.png)

Actual chain:

```text
sensor -> validate_data -> build_features -> train -> evaluate -> compare_with_champion(branch) -> register_model | skip_deploy -> finish
```

Gate rule:

```text
precision_at_10_new >= precision_at_10_champion
AND
precision_at_10_new >= threshold
```

Current evidence:

- DAG code: `../dags/nbo_retrain_dag.py`
- CLI/task evidence: `../reports/b07_dag_verification_log.md`
- UI graph screenshot: pending live capture (`../screenshots/b07_dag_graph.png` missing)

**Вывод:** DAG contract exists and parses; browser graph screenshot is still a capture TODO.


## 4. Airflow evidence / bad model gate

Expected screenshots:

- `../screenshots/b07_dag_run_success.png` - successful run
- `../screenshots/b07_gate_skip_deploy.png` - bad model blocked by gate

Actual current evidence from `../reports/b07_dag_verification_log.md`:

```text
NBO metric gate: precision_at_10_new=0.003204 precision_at_10_champion=0.003258 threshold=0.100000 decision=skip_deploy
Branch into skip_deploy

NBO metric gate: precision_at_10_new=0.200000 precision_at_10_champion=0.100000 threshold=0.100000 decision=register_model
Branch into register_model
```

**Вывод:** gate behavior is proven by CLI/task evidence; browser screenshots are pending because local browser tooling was absent.


## 5. IaC / Terraform / CI boundary

Terraform describes local stack manifests:

- storage manifest: batch path / bucket
- MLflow manifest: tracking URI / artifact store / aliases
- Airflow manifest: `dag_id=nbo_retrain_pipeline`
- API manifest: `/health`, `/score`, `/batch-score`, `/metrics`
- pipeline contract: feature list + gate rule + branch join

Evidence:

- Terraform code: `../infra/providers.tf`, `../infra/main.tf`, `../infra/variables.tf`, `../infra/outputs.tf`
- plan: `../reports/terraform_plan.txt`
- destroy plan: `../reports/terraform_destroy_plan.txt`
- CI: `../.github/workflows/ci.yml`

Boundary:

- CI -> install deps / compile / smoke tests / terraform fmt-validate-plan
- Airflow -> periodic train / evaluate / promote-or-skip
- training is not run in CI

**Вывод:** C3 is covered by IaC + compose + health evidence.


## 6. SLI/SLO 3 levels

Source: `../docs/sli_slo.md`. Thresholds are marked `[assumption]` until baseline evidence.

| level | SLI | normal | warning | critical | action |
|---|---|---|---|---|---|
| technical | `/health` availability | >= 99% demo window | 97-99% | < 97% | restart API / inspect alias / rollback |
| technical | p95 `/score` latency | < 500 ms cached | 500-800 ms | > 800 ms | check cache / scale / reduce IO |
| technical | API error-rate | < 1% | 1-3% | > 3% | rollback champion / inspect parity |
| model-data | validation critical checks | 100% pass | not_applicable | any fail | block train / fix batch |
| model-data | precision@10 vs champion | >= champion + threshold | equal | < champion | skip deploy / keep champion |
| model-data | drift share | < 0.20 | 0.20-0.40 | > 0.40 | inspect drift / retrain if stable |
| business | hit-rate@10 | >= baseline | -10% | -25% | review model / offer policy |
| business | scorable coverage | >= baseline | -5% | -15% | inspect ingestion / entity mapping |

**Вывод:** technical/model/business risks are mapped to observable SLI and incident actions.


## 7. MDD latency

Evidence:

- notebook: `../notebooks/mdd_latency.ipynb`
- result: `../reports/mdd_test_result.md`
- plot: `../reports/mdd_latency_distribution.png`

![MDD latency plot](../reports/mdd_latency_distribution.png)

Hypotheses:

- H0: `mean_improved >= mean_existing`
- H1: `mean_improved < mean_existing`

Test:

```python
scipy.stats.ttest_ind(improved, existing, equal_var=False, alternative="less")
```

Actual result:

- test: Welch t-test
- alpha: `0.05`
- statistic: `-1873.490`
- p-value: `0.00000000`
- robustness: Mann-Whitney U, p-value `0.00000000`
- decision: move heavy feature compute to batch + serve precomputed top-N from cache

**Вывод:** H0 rejected, cache/batch architecture accepted.


## 8. ADR decision

ADR file: `../adr/0001-latency-mdd-decision.md`

Decision summary:

- keep feature compute outside request path
- batch preprocessing + feature store/cache
- API `/score` serves precomputed top-N from cache
- p95 `/score` SLO target: `< 500 ms`
- monitoring needed: p95/p99 latency / stale cache / data freshness in DAG

Tradeoff:

- plus: lower p95/p99 / predictable request path
- minus: freshness lag / cache invalidation / DAG freshness monitoring

**Вывод:** MDD result is directly connected to serving architecture.


## 9. Evidence pack status

| item | status | evidence |
|---|---|---|
| docker ps healthy | pass | `../screenshots/docker_ps_healthy.png` |
| `/health` 200 | pass text + render | `../reports/api_smoke.md`, `../screenshots/health_200_render.png` |
| terraform plan | pass | `../reports/terraform_plan.txt` |
| MLflow alias | pending UI screenshot | `../reports/registry_demo.md`, blocker `../screenshots/mlflow_ui_screenshot_blocker.md` |
| Airflow run/gate screenshots | pending UI screenshot | `../reports/b07_dag_verification_log.md` |
| Grafana dashboard screenshot | pending UI screenshot | API evidence in `../reports/b08_*.json` |
| demo UI screenshot | pending browser screenshot | render substitute `../screenshots/demo_score_render.png` |

Full index: `../screenshots/INDEX.md`.

**Вывод:** backend/e2e command evidence exists; browser screenshots still need manual capture before final oral demo.


## 10. Final DoD checklist

| DoD / rubric | status | evidence |
|---|---|---|
| own scope done | pass | `manifest.md`, `README.md`, `HW_Design.ipynb`, `rubric_mapping.md`, `INDEX.md` |
| timestamp + masking | pass after lint | b12 verification |
| anti-leak check | pass after sanitize | `reports/sanitize_report.md` |
| synthetic boot | pass | `data/sample_synth.parquet`, `Makefile`, tests/reports |
| C1 goal/metric | pass | manifest sections 7-8 + metric report |
| C2 Level 2 maturity | pending | components exist, UI screenshots pending |
| C3 IaC + working service | pass | terraform plan + docker ps + `/health` 200 |
| C4 SLI/SLO | pass | `docs/sli_slo.md` |
| C5 MDD/ADR | pass | ADR + MDD report + plot |

**Вывод:** project is ready for homework review with one known gap class: live browser screenshots. Cloud deploy P2 is separate.
